# 00 · Download Sen1Floods11 (hand-labeled) to Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1ow1atency/floodfusion/blob/main/notebooks/00_download_data.ipynb)

Downloads the Sen1Floods11 v1.1 hand-labeled subset (~1.75 GB) to Drive, checks it, and builds a small sample zip for local CPU work. Safe to re-run: finished steps are skipped.

Mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Paths and bucket sources.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/floodfusion')
DATA_ROOT = DRIVE_ROOT / 'data' / 'sen1floods11'
SAMPLE_ZIP = DRIVE_ROOT / 'sen1floods11_sample.zip'
BUCKET = 'gs://sen1floods11/v1.1'
SOURCES = {
    'S1Hand': 'data/flood_events/HandLabeled/S1Hand',
    'S2Hand': 'data/flood_events/HandLabeled/S2Hand',
    'LabelHand': 'data/flood_events/HandLabeled/LabelHand',
    'splits': 'splits/flood_handlabeled',
}
KINDS = ['S1Hand', 'S2Hand', 'LabelHand']

DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('data root:', DATA_ROOT)

**Optional:** the bucket is public, so this cell does nothing by default. Set `USE_AUTH = True` and re-run it only if `gsutil` below asks for credentials.

In [ ]:
USE_AUTH = False  # set True only if gsutil asks for credentials

if USE_AUTH:
    from google.colab import auth
    auth.authenticate_user()
else:
    print('skipping auth (USE_AUTH = False)')

Download each folder with `gsutil -m rsync -r` (a few minutes); folders marked `.complete` are skipped.

In [ ]:
import subprocess

for name, src in SOURCES.items():
    dst = DATA_ROOT / name
    marker = dst / '.complete'
    if marker.exists():
        print(f'skip {name}: already downloaded')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    print(f'downloading {name} ...', flush=True)
    cmd = ['gsutil', '-m', 'rsync', '-r', f'{BUCKET}/{src}', str(dst)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'gsutil failed for {name}:\n{result.stderr[-2000:]}')
    marker.touch()
    print(f'done {name}')

File count and total size per folder.

In [ ]:
def list_files(folder):
    return [p for p in folder.iterdir() if p.is_file() and not p.name.startswith('.')]

for name in SOURCES:
    files = list_files(DATA_ROOT / name)
    size_mb = sum(p.stat().st_size for p in files) / 1e6
    print(f'{name:10s} {len(files):4d} files  {size_mb:8.1f} MB')

Chips per split CSV (no header; first column is `<chip_id>_S1Hand.tif`).

In [ ]:
import csv

def read_split(csv_path):
    with open(csv_path, newline='') as f:
        rows = [r for r in csv.reader(f) if r and r[0].strip()]
    return [r[0].strip().removesuffix('_S1Hand.tif') for r in rows]

splits = {p.stem: read_split(p) for p in sorted((DATA_ROOT / 'splits').glob('*.csv'))}
for name, chips in splits.items():
    print(f'{name:22s} {len(chips):4d} chips')

Check every chip in the CSVs has its S1, S2 and label file.

In [ ]:
present = {k: {p.name for p in list_files(DATA_ROOT / k)} for k in KINDS}
missing = [(split, chip, k)
           for split, chips in splits.items() for chip in chips for k in KINDS
           if f'{chip}_{k}.tif' not in present[k]]

if missing:
    print(f'{len(missing)} missing files:')
    for split, chip, k in missing:
        print(f'  [{split}] {chip}_{k}.tif')
else:
    print('OK: every chip in every split has S1Hand, S2Hand and LabelHand files.')

Build the sample zip (first 20 train + 5 Bolivia chips, all split CSVs); skipped if it already exists.

In [ ]:
import shutil, zipfile

N_TRAIN, N_BOLIVIA = 20, 5

if SAMPLE_ZIP.exists():
    print(f'skip: {SAMPLE_ZIP} already exists')
else:
    sample = splits['flood_train_data'][:N_TRAIN] + splits['flood_bolivia_data'][:N_BOLIVIA]
    tmp = Path('/content/sen1floods11_sample.zip')  # build locally, then copy, so Drive never holds a partial zip
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for chip in sample:
            for k in KINDS:
                zf.write(DATA_ROOT / k / f'{chip}_{k}.tif', f'{k}/{chip}_{k}.tif')
        for csv_path in sorted((DATA_ROOT / 'splits').glob('*.csv')):
            zf.write(csv_path, f'splits/{csv_path.name}')
    shutil.copy(tmp, SAMPLE_ZIP)
    print(f'wrote {SAMPLE_ZIP} ({len(sample)} chips)')

with zipfile.ZipFile(SAMPLE_ZIP) as zf:
    print(f'{len(zf.namelist())} entries, {SAMPLE_ZIP.stat().st_size / 1e6:.1f} MB')

Locally: download the zip from Drive and run `unzip sen1floods11_sample.zip -d data/sen1floods11` in the repo root.
The CSVs in the sample are the full splits, so most listed chips are not in the sample.